# Session V: File I/O and Structured Data

Persist data using text files, JSON, and CSV, and learn how to design file-based mini databases.

## Exercise 1: Daily Journal App
**Objective:** Practice reading, writing, and appending to text files.

Build a simple daily journal application that stores entries in a plain text file (`journal.txt`). Each entry should include a timestamp and the user's text.

**Tasks:**
1. `add_entry(filename, text)` → Append a new journal entry with the current date/time to the file.
2. `read_journal(filename)` → Read and display all entries from the file. Handle the case where the file doesn't exist yet.
3. `search_journal(filename, keyword)` → Read the file and return only lines that contain the keyword (case-insensitive).
4. `count_entries(filename)` → Return the total number of entries in the journal.
5. `run_journal()` → A menu-driven interface: [A]dd entry, [R]ead all, [S]earch, [C]ount, [Q]uit.

In [2]:
from datetime import datetime

JOURNAL_FILE = "journal.txt"


def add_entry(filename, text):
    """Append a journal entry with the current date and time."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    with open(filename, "a", encoding="utf-8") as file:
        file.write(f"[{timestamp}] {text}\n")


def read_journal(filename):
    """Read and display all journal entries."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            entries = file.readlines()

        if not entries:
            print("Your journal is empty.")
        else:
            for entry in entries:
                print(entry.strip())

    except FileNotFoundError:
        print("No journal found yet.")


def search_journal(filename, keyword):
    """Return lines containing the keyword, ignoring case."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            return [
                line.strip()
                for line in file
                if keyword.lower() in line.lower()
            ]

    except FileNotFoundError:
        return []


def count_entries(filename):
    """Return the total number of journal entries."""
    try:
        with open(filename, "r", encoding="utf-8") as file:
            return sum(1 for line in file if line.strip())

    except FileNotFoundError:
        return 0


def run_journal():
    """Run the menu-driven journal application."""
    while True:
        print("\n===== DAILY JOURNAL =====")
        print("[A]dd entry")
        print("[R]ead all entries")
        print("[S]earch entries")
        print("[C]ount entries")
        print("[Q]uit")

        choice = input("Choose an option: ").strip().upper()

        if choice == "A":
            text = input("Write your journal entry: ").strip()

            if text:
                add_entry(JOURNAL_FILE, text)
                print("Entry saved successfully!")
            else:
                print("Entry cannot be empty.")

        elif choice == "R":
            read_journal(JOURNAL_FILE)

        elif choice == "S":
            keyword = input("Enter a keyword: ").strip()

            if not keyword:
                print("Keyword cannot be empty.")
                continue

            results = search_journal(JOURNAL_FILE, keyword)

            if results:
                print("\nSearch results:")
                for result in results:
                    print(result)
            else:
                print("No matching entries found.")

        elif choice == "C":
            print(f"Total entries: {count_entries(JOURNAL_FILE)}")

        elif choice == "Q":
            print("Goodbye!")
            break

        else:
            print("Invalid option. Please choose A, R, S, C, or Q.")


# --- QUICK TEST ---
# Uncomment these lines if you want to test the functions separately.
add_entry(JOURNAL_FILE, "Started learning about file I/O today!")
add_entry(JOURNAL_FILE, "Python makes reading files so easy.")
read_journal(JOURNAL_FILE)
print("Search results:", search_journal(JOURNAL_FILE, "python"))
print("Total entries:", count_entries(JOURNAL_FILE))

# Run the interactive application:
run_journal()


[2026-10-22 11:26:03] Started learning about file I/O today!
[2026-10-22 11:26:03] Python makes reading files so easy.
[2026-10-22 11:27:53] Started learning about file I/O today!
[2026-10-22 11:27:53] Python makes reading files so easy.
[2026-10-22 11:34:35] Started learning about file I/O today!
[2026-10-22 11:34:35] Python makes reading files so easy.
Search results: ['[2026-10-22 11:26:03] Python makes reading files so easy.', '[2026-10-22 11:27:53] Python makes reading files so easy.', '[2026-10-22 11:34:35] Python makes reading files so easy.']
Total entries: 6

===== DAILY JOURNAL =====
[A]dd entry
[R]ead all entries
[S]earch entries
[C]ount entries
[Q]uit
Entry saved successfully!

===== DAILY JOURNAL =====
[A]dd entry
[R]ead all entries
[S]earch entries
[C]ount entries
[Q]uit
[2026-10-22 11:26:03] Started learning about file I/O today!
[2026-10-22 11:26:03] Python makes reading files so easy.
[2026-10-22 11:27:53] Started learning about file I/O today!
[2026-10-22 11:27:53] Py

---
## Exercise 2: To-Do List Manager (JSON Persistence)
**Objective:** Build a CRUD application that stores data in a JSON file.

Create a to-do list manager where tasks are stored as a list of dictionaries in a `tasks.json` file. Each task has a `title`, a `priority` (high/medium/low), and a `done` status (True/False).

**Tasks:**
1. `load_tasks(filename)` → Load tasks from JSON. Return an empty list if the file is missing or corrupted.
2. `save_tasks(filename, tasks)` → Save the tasks list to JSON with pretty formatting.
3. `add_task(tasks)` → Prompt for title and priority, append a new task dict (done=False).
4. `list_tasks(tasks)` → Display all tasks in a formatted table with their index, status, title, and priority.
5. `mark_done(tasks)` → Ask for a task index and set its `done` field to True.
6. `delete_task(tasks)` → Ask for a task index and remove the task from the list.
7. `run_todo()` → Menu loop: [A]dd, [L]ist, [D]one, [R]emove, [Q]uit. Auto-save after every change.

**Data structure:**
```python
[
    {"title": "Buy groceries", "priority": "high", "done": False},
    {"title": "Read chapter 5", "priority": "medium", "done": True}
]
```

In [14]:

import json

TODO_FILE = "tasks.json"


def load_tasks(filename):
    """Load tasks from JSON. Return [] if missing or corrupted."""
    try:
        with open(filename, "r", encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []


def save_tasks(filename, tasks):
    """Save the tasks list to JSON with pretty formatting."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(tasks, f, indent=2)


def add_task(tasks):
    """Prompt for title and priority, then append a new task."""
    title = input("Enter task title: ").strip()

    if not title:
        print("Title cannot be empty.")
        return False

    priority = input("Enter priority (high/medium/low): ").strip().lower()

    if priority not in ["high", "medium", "low"]:
        print("Invalid priority. Please enter high, medium, or low.")
        return False

    tasks.append({
        "title": title,
        "priority": priority,
        "done": False
    })

    print("Task added successfully!")
    return True


def list_tasks(tasks):
    """Display tasks with index, status, title, and priority."""
    if not tasks:
        print("No tasks to display.")
        return

    print(f"\n{'Index':<8}{'Status':<10}{'Title':<30}{'Priority'}")
    print("-" * 65)

    for i, task in enumerate(tasks):
        status = "Done" if task["done"] else "Pending"
        print(
            f"{i:<8}{status:<10}{task['title']:<30}"
            f"{task['priority']}"
        )


def mark_done(tasks):
    """Ask for a task index and mark it as done."""
    if not tasks:
        print("No tasks available.")
        return False

    try:
        index = int(input("Enter task index: "))

        if 0 <= index < len(tasks):
            if tasks[index]["done"]:
                print("Task is already completed.")
                return False

            tasks[index]["done"] = True
            print("Task marked as done!")
            return True

        print("Invalid task index.")
        return False

    except ValueError:
        print("Please enter a valid number.")
        return False


def delete_task(tasks):
    """Ask for a task index and remove it using pop()."""
    if not tasks:
        print("No tasks available to remove.")
        return False

    try:
        index = int(input("Enter task index to delete: "))

        if 0 <= index < len(tasks):
            removed_task = tasks.pop(index)
            print(f"Removed task: {removed_task['title']}")
            return True

        print("Invalid task index.")
        return False

    except ValueError:
        print("Please enter a valid number.")
        return False


def run_todo():
    """Run the menu loop and automatically save changes."""
    tasks = load_tasks(TODO_FILE)

    while True:
        print("\n--- To-Do List ---")
        print("[A]dd task")
        print("[L]ist tasks")
        print("[D]one task")
        print("[R]emove task")
        print("[Q]uit")

        choice = input("Enter your choice: ").strip().lower()

        if choice == "a":
            if add_task(tasks):
                save_tasks(TODO_FILE, tasks)

        elif choice == "l":
            list_tasks(tasks)

        elif choice == "d":
            if mark_done(tasks):
                save_tasks(TODO_FILE, tasks)

        elif choice == "r":
            if delete_task(tasks):
                save_tasks(TODO_FILE, tasks)

        elif choice == "q":
            save_tasks(TODO_FILE, tasks)
            print("Tasks saved. Goodbye!")
            break

        else:
            print("Invalid choice. Please try again.")


# --- QUICK TEST ---
# Test JSON loading, saving, and displaying without adding duplicates.
tasks = load_tasks(TODO_FILE)
save_tasks(TODO_FILE, tasks)
tasks = load_tasks(TODO_FILE)
list_tasks(tasks)

# Uncomment to launch the interactive menu:
run_todo()



Index   Status    Title                         Priority
-----------------------------------------------------------------
0       Pending   Test task                     high
1       Done      Buy groceries                 high

--- To-Do List ---
[A]dd task
[L]ist tasks
[D]one task
[R]emove task
[Q]uit


Task added successfully!

--- To-Do List ---
[A]dd task
[L]ist tasks
[D]one task
[R]emove task
[Q]uit
Removed task: Test task

--- To-Do List ---
[A]dd task
[L]ist tasks
[D]one task
[R]emove task
[Q]uit
Tasks saved. Goodbye!


In [15]:
tasks = load_tasks(TODO_FILE)
list_tasks(tasks)


Index   Status    Title                         Priority
-----------------------------------------------------------------
0       Done      Buy groceries                 high
1       Pending   complete python assignment    high


---
## Exercise 3: CSV Expense Importer
**Objective:** Read, validate, and analyze data from a CSV file.

You receive a CSV file of monthly expenses. Some rows may have missing or invalid data. Write functions to import the data safely, clean it, and produce summary reports.

**First, run the cell below to generate the sample CSV file.** Then write the functions to process it.

**Tasks:**
1. `load_expenses(filename)` → Read the CSV using `csv.DictReader`. Validate each row: skip rows where `amount` is not a valid number or `category` is empty. Return a list of clean dicts (with `amount` converted to float).
2. `total_by_category(expenses)` → Return a dict mapping each category to its total spending.
3. `most_expensive(expenses)` → Return the single most expensive entry.
4. `print_report(expenses)` → Print a formatted summary: total spent, breakdown by category, and the most expensive item.

In [16]:
import csv

# --- RUN THIS CELL FIRST to generate the sample CSV ---
sample_data = """date,description,category,amount
2024-01-05,Grocery store,Food,52.30
2024-01-06,Bus pass,Transport,40.00
2024-01-07,Coffee shop,Food,4.50
2024-01-08,Netflix,Entertainment,15.99
2024-01-09,Gas station,Transport,bad_data
2024-01-10,Restaurant,Food,38.75
2024-01-11,Gym membership,,45.00
2024-01-12,Bookstore,Education,22.50
2024-01-13,Electricity bill,Utilities,89.00
2024-01-14,Movie tickets,Entertainment,24.00
2024-01-15,Taxi,Transport,18.50
2024-01-16,Online course,Education,49.99
2024-01-17,Supermarket,Food,67.20
2024-01-18,Water bill,Utilities,not_a_number
2024-01-19,Concert,Entertainment,75.00
2024-01-20,Pharmacy,Health,12.40"""

with open("expenses.csv", "w", newline="") as f:
    f.write(sample_data)

print("✅ expenses.csv created successfully.")

✅ expenses.csv created successfully.


In [17]:

import csv


def load_expenses(filename):
    """Read the CSV, skip invalid rows, and return clean expense dictionaries."""
    expenses = []

    try:
        with open(filename, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            for row in reader:
                # Skip rows with a missing or empty category
                category = (row.get("category") or "").strip()
                if not category:
                    continue

                # Skip rows where amount is missing or not a valid number
                try:
                    amount = float(row.get("amount", ""))
                except (ValueError, TypeError):
                    continue

                # Keep the cleaned category and numeric amount
                row["category"] = category
                row["amount"] = amount
                expenses.append(row)

    except FileNotFoundError:
        print(f"Error: File '{filename}' was not found.")

    return expenses


def total_by_category(expenses):
    """Return a dictionary mapping each category to total spending."""
    totals = {}

    for expense in expenses:
        category = expense["category"]
        amount = expense["amount"]

        totals[category] = totals.get(category, 0.0) + amount

    return totals


def most_expensive(expenses):
    """Return the single expense with the highest amount."""
    if not expenses:
        return None

    return max(expenses, key=lambda expense: expense["amount"])


def print_report(expenses):
    """Print total spending, category totals, and the most expensive item."""
    print("\n========== MONTHLY EXPENSE REPORT ==========")

    if not expenses:
        print("No valid expenses found.")
        print("============================================")
        return

    # Calculate total spending
    total_spent = sum(expense["amount"] for expense in expenses)
    print(f"Total spent: ${total_spent:,.2f}")

    # Display spending by category
    print("\nSpending by category:")
    category_totals = total_by_category(expenses)

    for category, total in sorted(category_totals.items()):
        print(f"  {category:<16} ${total:,.2f}")

    # Display the most expensive expense
    expensive = most_expensive(expenses)

    print("\nMost expensive item:")
    print(f"  Description: {expensive['description']}")
    print(f"  Category:    {expensive['category']}")
    print(f"  Amount:      ${expensive['amount']:,.2f}")
    print(f"  Date:        {expensive['date']}")

    print("============================================")


# --- TEST ---
expenses = load_expenses("expenses.csv")
print_report(expenses)



========== MONTHLY EXPENSE REPORT ==========
Total spent: $510.13

Spending by category:
  Education        $72.49
  Entertainment    $114.99
  Food             $162.75
  Health           $12.40
  Transport        $58.50
  Utilities        $89.00

Most expensive item:
  Description: Electricity bill
  Category:    Utilities
  Amount:      $89.00
  Date:        2024-01-13


---
## Exercise 4: File Content Validator
**Objective:** Write robust validation functions for both JSON and CSV files.

You will build a validation toolkit that checks whether files are well-formed and contain the expected structure *before* your application tries to use the data.

**First, run the cell below to generate sample test files** (some valid, some intentionally broken). Then write the validators.

**Tasks:**
1. `validate_json(filename, required_keys)` → Check that the file (a) exists, (b) contains valid JSON, (c) is a list of dicts, and (d) each dict contains all `required_keys`. Return a report dict: `{"valid": int, "invalid": int, "errors": [str]}`.
2. `validate_csv(filename, expected_columns)` → Check that the file (a) exists, (b) has the correct number of columns in the header, and (c) each data row has the correct number of fields. Return a similar report dict.
3. `print_validation_report(report, filename)` → Pretty-print the validation results.

In [19]:
import json

# --- RUN THIS CELL FIRST to generate test files ---

# Good JSON file
good_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25, "email": "bob@test.com"},
    {"name": "Charlie", "age": 30, "email": "charlie@test.com"},
]
with open("good_data.json", "w") as f:
    json.dump(good_json, f, indent=2)

# Bad JSON file (missing keys, wrong types)
bad_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25},
    "this is not a dict",
    {"name": "Diana", "age": "not_a_number", "email": "diana@test.com"},
]
with open("bad_data.json", "w") as f:
    json.dump(bad_json, f, indent=2)

# Corrupted JSON file
with open("corrupted.json", "w") as f:
    f.write('{"name": "broken", missing_quotes: true}')

# Good CSV file
with open("good_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25,bob@test.com\n")

# Bad CSV file (inconsistent columns)
with open("bad_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25\n")
    f.write("Charlie,30,charlie@test.com,extra_field\n")
    f.write("Diana,28,diana@test.com\n")

print("✅ All test files created successfully.")

✅ All test files created successfully.


In [20]:

import json
import csv


def validate_json(filename, required_keys):
    """Validate JSON existence, syntax, root structure, and required keys."""
    report = {"valid": 0, "invalid": 0, "errors": []}

    # Check whether the file exists and contains valid JSON
    try:
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        report["errors"].append(f"File not found: {filename}")
        report["invalid"] = 1
        return report
    except json.JSONDecodeError as e:
        report["errors"].append(f"Invalid JSON syntax: {e}")
        report["invalid"] = 1
        return report

    # The root must be a list
    if not isinstance(data, list):
        report["errors"].append("JSON root must be a list.")
        report["invalid"] = 1
        return report

    # Validate each item in the list
    for index, item in enumerate(data):
        if not isinstance(item, dict):
            report["invalid"] += 1
            report["errors"].append(
                f"Item {index}: expected a dictionary, got {type(item).__name__}."
            )
            continue

        missing_keys = [key for key in required_keys if key not in item]

        if missing_keys:
            report["invalid"] += 1
            report["errors"].append(
                f"Item {index}: missing required keys: {', '.join(missing_keys)}."
            )
        else:
            report["valid"] += 1

    return report


def validate_csv(filename, expected_columns):
    """Validate CSV existence, header column count, and row field counts."""
    report = {"valid": 0, "invalid": 0, "errors": []}

    try:
        with open(filename, "r", newline="", encoding="utf-8") as f:
            reader = csv.reader(f)

            # Check that a header exists
            header = next(reader, None)

            if header is None:
                report["errors"].append("CSV file is empty; header is missing.")
                report["invalid"] = 1
                return report

            # Check the number of columns in the header
            if len(header) != expected_columns:
                report["errors"].append(
                    f"Header has {len(header)} columns; "
                    f"expected {expected_columns}."
                )
                report["invalid"] = 1
                return report

            # Check every data row
            for line_number, row in enumerate(reader, start=2):
                if len(row) == expected_columns:
                    report["valid"] += 1
                else:
                    report["invalid"] += 1
                    report["errors"].append(
                        f"Line {line_number}: found {len(row)} fields; "
                        f"expected {expected_columns}."
                    )

    except FileNotFoundError:
        report["errors"].append(f"File not found: {filename}")
        report["invalid"] = 1

    return report


def print_validation_report(report, filename):
    """Print a readable validation summary and all errors."""
    print(f"\nValidation report: {filename}")
    print("-" * 50)
    print(f"Valid records:   {report['valid']}")
    print(f"Invalid records: {report['invalid']}")

    if report["errors"]:
        print("Errors:")
        for error in report["errors"]:
            print(f"  - {error}")
    else:
        print("No errors found.")

    print("-" * 50)


# --- TEST ---
print("=" * 50)
report = validate_json("good_data.json", ["name", "age", "email"])
print_validation_report(report, "good_data.json")

print("=" * 50)
report = validate_json("bad_data.json", ["name", "age", "email"])
print_validation_report(report, "bad_data.json")

print("=" * 50)
report = validate_json("corrupted.json", ["name", "age", "email"])
print_validation_report(report, "corrupted.json")

print("=" * 50)
report = validate_csv("good_data.csv", 3)
print_validation_report(report, "good_data.csv")

print("=" * 50)
report = validate_csv("bad_data.csv", 3)
print_validation_report(report, "bad_data.csv")



Validation report: good_data.json
--------------------------------------------------
Valid records:   3
Invalid records: 0
No errors found.
--------------------------------------------------

Validation report: bad_data.json
--------------------------------------------------
Valid records:   2
Invalid records: 2
Errors:
  - Item 1: missing required keys: email.
  - Item 2: expected a dictionary, got str.
--------------------------------------------------

Validation report: corrupted.json
--------------------------------------------------
Valid records:   0
Invalid records: 1
Errors:
  - Invalid JSON syntax: Expecting property name enclosed in double quotes: line 1 column 20 (char 19)
--------------------------------------------------

Validation report: good_data.csv
--------------------------------------------------
Valid records:   2
Invalid records: 0
No errors found.
--------------------------------------------------

Validation report: bad_data.csv
------------------------------

---
## Exercise 5: Format Converter & Exporter
**Objective:** Convert data between JSON and CSV formats in both directions.

Build a bidirectional converter that can transform data files between JSON and CSV. This is an extremely common real-world task — APIs return JSON, but spreadsheets need CSV, and vice versa.

**Tasks:**
1. `json_to_csv(json_filename, csv_filename)` → Read a JSON file (list of dicts) and write it as a CSV file. Use the keys of the first dict as column headers.
2. `csv_to_json(csv_filename, json_filename)` → Read a CSV file and write it as a JSON file (list of dicts).
3. `export_filtered_csv(json_filename, csv_filename, filter_key, filter_value)` → Read a JSON file, keep only records where `record[filter_key] == filter_value`, and export the filtered data to CSV.
4. Handle edge cases: empty files, missing files, and mismatched structures.

In [18]:
import json
import csv

# --- Setup: create a test JSON file ---
students = [
    {"name": "Alice", "age": 22, "city": "Madrid", "grade": "A"},
    {"name": "Bob", "age": 25, "city": "London", "grade": "B"},
    {"name": "Charlie", "age": 30, "city": "Paris", "grade": "A"},
    {"name": "Diana", "age": 28, "city": "Madrid", "grade": "C"},
    {"name": "Eve", "age": 23, "city": "London", "grade": "A"},
]

with open("students.json", "w") as f:
    json.dump(students, f, indent=2)

print("✅ students.json created.")

✅ students.json created.


In [ ]:

import json
import csv


def json_to_csv(json_filename, csv_filename):
    """Convert a JSON file (list of dictionaries) to a CSV file."""
    try:
        with open(json_filename, "r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, list):
            print("Error: JSON data must be a list.")
            return

        if not data:
            print("No data to export: JSON list is empty.")
            return

        if not all(isinstance(record, dict) for record in data):
            print("Error: Every JSON record must be a dictionary.")
            return

        # Use keys from the first dictionary as CSV headers
        fieldnames = list(data[0].keys())

        with open(csv_filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(
                f, fieldnames=fieldnames, extrasaction="ignore"
            )
            writer.writeheader()
            writer.writerows(data)

        print(f"Successfully converted {json_filename} to {csv_filename}.")

    except FileNotFoundError:
        print(f"Error: File '{json_filename}' was not found.")
    except json.JSONDecodeError:
        print(f"Error: '{json_filename}' contains invalid JSON.")
    except OSError as e:
        print(f"File error: {e}")


def csv_to_json(csv_filename, json_filename):
    """Convert a CSV file to a JSON file (list of dictionaries)."""
    try:
        with open(csv_filename, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            if reader.fieldnames is None:
                print("Error: CSV file is empty or has no header.")
                return

            # Reading all rows as dictionaries
            data = list(reader)

        with open(json_filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

        print(f"Successfully converted {csv_filename} to {json_filename}.")

    except FileNotFoundError:
        print(f"Error: File '{csv_filename}' was not found.")
    except OSError as e:
        print(f"File error: {e}")


def export_filtered_csv(json_filename, csv_filename, filter_key, filter_value):
    """Export only JSON records matching the requested key and value."""
    try:
        with open(json_filename, "r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, list):
            print("Error: JSON data must be a list.")
            return

        if not all(isinstance(record, dict) for record in data):
            print("Error: Every JSON record must be a dictionary.")
            return

        # Keeping records where the key exists and the value matches
        filtered = [
            record for record in data
            if record.get(filter_key) == filter_value
        ]

        if not filtered:
            print(f"No records found where {filter_key} = {filter_value}.")
            return

        # Keeping the first record's key order for the CSV columns
        fieldnames = list(filtered[0].keys())

        with open(csv_filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(
                f, fieldnames=fieldnames, extrasaction="ignore"
            )
            writer.writeheader()
            writer.writerows(filtered)

        print(
            f"Exported {len(filtered)} matching records "
            f"to {csv_filename}."
        )

    except FileNotFoundError:
        print(f"Error: File '{json_filename}' was not found.")
    except json.JSONDecodeError:
        print(f"Error: '{json_filename}' contains invalid JSON.")
    except OSError as e:
        print(f"File error: {e}")


# --- TEST ---
# JSON -> CSV
json_to_csv("students.json", "students_export.csv")

# CSV -> JSON 
csv_to_json("students_export.csv", "students_roundtrip.json")

# Filtered export: only students from Madrid
export_filtered_csv(
    "students.json", "madrid_students.csv", "city", "Madrid"
)

# Verify round-trip
with open("students_roundtrip.json", "r", encoding="utf-8") as f:
    roundtrip = json.load(f)

print(f"\nRound-trip data ({len(roundtrip)} records):")
for r in roundtrip:
    print(f"  {r}")


Successfully converted students.json to students_export.csv.
Successfully converted students_export.csv to students_roundtrip.json.
Exported 2 matching records to madrid_students.csv.

Round-trip data (5 records):
  {'name': 'Alice', 'age': '22', 'city': 'Madrid', 'grade': 'A'}
  {'name': 'Bob', 'age': '25', 'city': 'London', 'grade': 'B'}
  {'name': 'Charlie', 'age': '30', 'city': 'Paris', 'grade': 'A'}
  {'name': 'Diana', 'age': '28', 'city': 'Madrid', 'grade': 'C'}
  {'name': 'Eve', 'age': '23', 'city': 'London', 'grade': 'A'}


In [22]:
with open("madrid_students.csv", "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)

{'name': 'Alice', 'age': '22', 'city': 'Madrid', 'grade': 'A'}
{'name': 'Diana', 'age': '28', 'city': 'Madrid', 'grade': 'C'}


---
## Exercise 6: Student Records Database (Capstone)
**Objective:** Build a complete file-based CRUD application combining JSON persistence, CSV import/export, and data validation.

Create a student records management system that stores data in a JSON file and supports importing/exporting CSV. This exercise combines **every concept** from this session.

**Required features:**
1. **[A]dd Student** — Prompt for name, age, email, and grade. Validate all inputs (age must be a positive integer, email must contain `@`, grade must be A/B/C/D/F). Store in JSON.
2. **[L]ist Students** — Display all students in a formatted table.
3. **[S]earch** — Find a student by name (case-insensitive partial match).
4. **[U]pdate** — Modify a student's grade or email by name.
5. **[D]elete** — Remove a student by name.
6. **[I]mport CSV** — Load students from a CSV file, validating each row before adding. Skip invalid rows and report what was skipped.
7. **[E]xport CSV** — Export all current students to a CSV file.
8. **[Q]uit** — Exit the program.

**Data structure (JSON):**
```python
[
    {"name": "Alice", "age": 22, "email": "alice@uni.edu", "grade": "A"},
    {"name": "Bob", "age": 25, "email": "bob@uni.edu", "grade": "B"}
]
```

In [23]:

import json
import csv

DB_FILE = "students_db.json"

# --- PERSISTENCE ---

def load_db(filename):
    """Load student records from JSON. Return [] if missing or corrupted."""
    try:
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list) and all(isinstance(s, dict) for s in data):
            return data

        print("Warning: Invalid database structure. Starting with an empty database.")
        return []

    except (FileNotFoundError, json.JSONDecodeError):
        return []


def save_db(filename, students):
    """Save student records to JSON."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(students, f, indent=2)


# --- VALIDATION ---

def validate_student(name, age, email, grade):
    """Validate student fields. Return (is_valid, error_message)."""
    name = str(name).strip()
    email = str(email).strip()
    grade = str(grade).strip().upper()

    if not name:
        return False, "Name cannot be empty."

    try:
        age_number = int(age)
        if str(age).strip() != str(age_number) or age_number <= 0:
            return False, "Age must be a positive integer."
    except (ValueError, TypeError):
        return False, "Age must be a positive integer."

    if "@" not in email:
        return False, "Email must contain '@'."

    if grade not in ["A", "B", "C", "D", "F"]:
        return False, "Grade must be A, B, C, D, or F."

    return True, ""


# --- CRUD OPERATIONS ---

def add_student(students):
    """Prompt for student info, validate, and add to list."""
    name = input("Student name: ").strip()
    age = input("Age: ").strip()
    email = input("Email: ").strip()
    grade = input("Grade (A/B/C/D/F): ").strip().upper()

    is_valid, error = validate_student(name, age, email, grade)

    if not is_valid:
        print("Cannot add student:", error)
        return False

    students.append({
        "name": name,
        "age": int(age),
        "email": email,
        "grade": grade
    })
    print(f"Student '{name}' added successfully.")
    return True


def list_students(students):
    """Display all students in a formatted table."""
    if not students:
        print("No students in the database.")
        return

    print()
    print(f"{'No.':<5}{'Name':<20}{'Age':<8}{'Email':<28}{'Grade'}")
    print("-" * 70)

    for index, student in enumerate(students, start=1):
        print(
            f"{index:<5}{student['name']:<20}"
            f"{student['age']:<8}{student['email']:<28}"
            f"{student['grade']}"
        )


def search_student(students):
    """Search by name using a case-insensitive partial match."""
    term = input("Enter name or part of a name to search: ").strip().lower()

    if not term:
        print("Search term cannot be empty.")
        return

    matches = [
        student for student in students
        if term in student["name"].lower()
    ]

    if matches:
        print(f"Found {len(matches)} matching student(s):")
        list_students(matches)
    else:
        print("No matching students found.")


def update_student(students):
    """Update a student's grade or email by name."""
    name = input("Enter the student's name to update: ").strip().lower()

    matches = [
        student for student in students
        if student["name"].lower() == name
    ]

    if not matches:
        print("Student not found.")
        return False

    if len(matches) > 1:
        print("Multiple students have that name. Please use a unique name.")
        return False

    student = matches[0]
    field = input("Update [G]rade or [E]mail? ").strip().lower()

    if field == "g":
        new_grade = input("New grade (A/B/C/D/F): ").strip().upper()
        valid, error = validate_student(
            student["name"], student["age"], student["email"], new_grade
        )
        if not valid:
            print("Update failed:", error)
            return False
        student["grade"] = new_grade

    elif field == "e":
        new_email = input("New email: ").strip()
        valid, error = validate_student(
            student["name"], student["age"], new_email, student["grade"]
        )
        if not valid:
            print("Update failed:", error)
            return False
        student["email"] = new_email

    else:
        print("Invalid field choice.")
        return False

    print(f"Student '{student['name']}' updated successfully.")
    return True


def delete_student(students):
    """Remove a student by name after confirmation."""
    name = input("Enter the student's name to delete: ").strip().lower()

    matches = [
        student for student in students
        if student["name"].lower() == name
    ]

    if not matches:
        print("Student not found.")
        return False

    if len(matches) > 1:
        print("Multiple students have that name. Please use a unique name.")
        return False

    student = matches[0]
    confirm = input(
        f"Delete {student['name']}? Type 'yes' to confirm: "
    ).strip().lower()

    if confirm == "yes":
        students.remove(student)
        print(f"Student '{student['name']}' deleted.")
        return True

    print("Deletion cancelled.")
    return False


# --- IMPORT / EXPORT ---

def import_csv(students, csv_filename):
    """Import CSV students, validate each row, and report skipped rows."""
    required_columns = {"name", "age", "email", "grade"}

    try:
        with open(csv_filename, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            if reader.fieldnames is None or not required_columns.issubset(
                set(reader.fieldnames)
            ):
                print("Import failed: CSV must contain name, age, email, grade.")
                return

            added = 0
            skipped = 0

            for line_number, row in enumerate(reader, start=2):
                name = row.get("name")
                age = row.get("age")
                email = row.get("email")
                grade = row.get("grade")

                valid, error = validate_student(name, age, email, grade)

                if not valid:
                    print(f"Skipped CSV line {line_number}: {error}")
                    skipped += 1
                    continue

                students.append({
                    "name": name.strip(),
                    "age": int(age),
                    "email": email.strip(),
                    "grade": grade.strip().upper()
                })
                added += 1

            print(f"Import complete: {added} added, {skipped} skipped.")

    except FileNotFoundError:
        print(f"Import failed: File '{csv_filename}' not found.")
    except OSError as e:
        print(f"Import failed: {e}")


def export_csv(students, csv_filename):
    """Export all students to CSV."""
    fieldnames = ["name", "age", "email", "grade"]

    try:
        with open(csv_filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(students)

        print(f"Exported {len(students)} student(s) to '{csv_filename}'.")

    except OSError as e:
        print(f"Export failed: {e}")


# --- MAIN MENU ---

def run_student_db():
    """Run the student records database menu."""
    students = load_db(DB_FILE)
    print(f"Loaded {len(students)} student(s) from database.")

    while True:
        print("\n===== STUDENT RECORDS DATABASE =====")
        print("[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete")
        print("[I]mport CSV  [E]xport CSV  [Q]uit")

        choice = input("> ").strip().lower()

        if choice == "a":
            if add_student(students):
                save_db(DB_FILE, students)

        elif choice == "l":
            list_students(students)

        elif choice == "s":
            search_student(students)

        elif choice == "u":
            if update_student(students):
                save_db(DB_FILE, students)

        elif choice == "d":
            if delete_student(students):
                save_db(DB_FILE, students)

        elif choice == "i":
            csv_file = input("CSV filename to import: ").strip()
            before = len(students)
            import_csv(students, csv_file)
            if len(students) != before:
                save_db(DB_FILE, students)

        elif choice == "e":
            csv_file = input("CSV filename to export to: ").strip()
            export_csv(students, csv_file)

        elif choice == "q":
            save_db(DB_FILE, students)
            print("Database saved. Goodbye!")
            break

        else:
            print("Invalid option. Please try again.")


In [24]:
# --- Generate a sample CSV for testing the Import feature ---

sample_import = """name,age,email,grade
Frank,21,frank@uni.edu,B
Grace,23,grace@uni.edu,A
Hank,not_a_number,hank@uni.edu,C
Ivy,20,ivy_no_at_sign,A
Jack,22,jack@uni.edu,Z
Karen,24,karen@uni.edu,B"""

with open("import_students.csv", "w", newline="") as f:
    f.write(sample_import)

print("import_students.csv created (3 valid, 3 invalid rows).")

import_students.csv created (3 valid, 3 invalid rows).


In [25]:
run_student_db()

Loaded 0 student(s) from database.

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit
Skipped CSV line 4: Age must be a positive integer.
Skipped CSV line 5: Email must contain '@'.
Skipped CSV line 6: Grade must be A, B, C, D, or F.
Import complete: 3 added, 3 skipped.

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit

No.  Name                Age     Email                       Grade
----------------------------------------------------------------------
1    Frank               21      frank@uni.edu               B
2    Grace               23      grace@uni.edu               A
3    Karen               24      karen@uni.edu               B

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit
Found 1 matching student(s):

No.  Name                Age     Email                       Grade
----

In [28]:
with open("students_export.csv", "r", encoding="utf-8") as f:
    print(f.read())

name,age,email,grade
Frank,21,frank@uni.edu,B
Grace,23,grace@uni.edu,C
Karen,24,karen@uni.edu,B

